In [2]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import os
from ast import literal_eval
from collections import Counter
import math
import numpy as np
import seaborn as sns

In [3]:
username = os.path.expanduser('~')
username

'/home/ehdgus7279'

In [4]:
benchmarks = [
    # Low-sharing Apps
    "fdtd2d",
    "jacobi1d",
    "fastwalshtransform",
    "relu",
    "fft",
    "fir",
    "sssp",
    "convolution2d",
    "color",
    "bitonicsort",
    "pagerank",
    "simpleconvolution",

    # High-sharing Apps
    'gups',    
    'syr2k',
    'spmv',
    'mis', 
    'stencil2d',
    'gemm',
    'lu',
    'matrixmultiplication',
    'mm2',
    'mm3',
    'gaussian',
    'matrixtranspose',
    'im2col',
    'vgg16',
    'bfs',
]

In [5]:
num_chiplets = 36
ns = 0.000000001
chip_cols = [str(i) for i in range(36)]

In [11]:
base_dir=f'{username}/mgvm/results/memory-traces'
date = "20260221-223953"
expanded_dir = f"{base_dir}/{date}/expanded"
metrics_dir = f"../pc-rdma-metrics"
trace_dir = f"{base_dir}/{date}"
print(f"{base_dir}/{date}")

/home/ehdgus7279/mgvm/results/memory-traces/20260221-223953


In [12]:
benchmark = 'syr2k'
trace_file = f"{metrics_dir}/{benchmark}_pc_rdma_metrics.csv"

df = pd.read_csv(trace_file)
df

,interval,pc,ro_shared_cacheline_count,ro_shared_access_count,ro_shared_access_mean,ro_shared_access_std,ro_shared_shared_degree_mean,ro_shared_shared_degree_std,interval_total_cachelines_x,ro_shared_cacheline_ratio,...,private_access_mean,private_access_std,private_shared_degree_mean,private_shared_degree_std,interval_total_cachelines,private_cacheline_ratio,interval_total_access,private_access_ratio,pc_total_cachelines,pc_total_access
0,0.0,571330968,13388.0,17534.0,1.309680,0.462379,1.016657,0.127986,16015.0,0.835966,...,1.329143,0.469991,0.0,0.0,16015.0,0.163909,21583.0,0.161655,16013,21023
1,0.0,571330816,1.0,288.0,288.000000,0.000000,35.000000,0.000000,16015.0,0.000062,...,0.000000,0.000000,0.0,0.0,NaN,0.000000,NaN,0.000000,1,288
2,0.0,571330836,1.0,272.0,272.000000,0.000000,33.000000,0.000000,16015.0,0.000062,...,0.000000,0.000000,0.0,0.0,NaN,0.000000,NaN,0.000000,1,272
3,10000.0,571330968,13332.0,17099.0,1.282553,0.450258,1.009451,0.096759,18096.0,0.736737,...,0.000000,0.000000,0.0,0.0,NaN,0.000000,NaN,0.000000,14648,18871
4,10000.0,571331132,20.0,63.0,3.150000,0.366348,1.000000,0.000000,18096.0,0.001105,...,4.251534,1.641828,0.0,0.0,18096.0,0.009008,25750.0,0.026913,183,756
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16578,16490000.0,571331160,0.0,0.0,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,...,1.000000,0.000000,0.0,0.0,14890.0,0.000067,17257.0,0.000058,1,1
16579,16490000.0,571331168,0.0,0.0,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,...,1.000000,0.000000,0.0,0.0,14890.0,0.000336,17257.0,0.000290,5,5
16580,16490000.0,571331176,0.0,0.0,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,...,1.000000,0.000000,0.0,0.0,14890.0,0.000067,17257.0,0.000058,1,1
16581,16490000.0,571331184,0.0,0.0,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,...,1.000000,0.000000,0.0,0.0,14890.0,0.000067,17257.0,0.000058,1,1


In [16]:
df.groupby(['interval']).agg({
    'ro_shared_cacheline_count': 'sum',
    'rw_shared_cacheline_count': 'sum',
    'private_cacheline_count': 'sum',
    'ro_shared_access_count': 'sum',
    'rw_shared_access_count': 'sum',
    'private_access_count': 'sum',
}).reset_index()

,interval,ro_shared_cacheline_count,rw_shared_cacheline_count,private_cacheline_count,ro_shared_access_count,rw_shared_access_count,private_access_count
0,0.0,13390.0,0.0,2625.0,18094.0,0.0,3489.0
1,10000.0,13369.0,3456.0,3273.0,17179.0,4103.0,4468.0
2,20000.0,14227.0,264.0,2413.0,18196.0,274.0,5143.0
3,30000.0,18778.0,5655.0,2245.0,23681.0,6824.0,2605.0
4,40000.0,13138.0,17785.0,1082.0,17738.0,20466.0,4288.0
...,...,...,...,...,...,...,...
1645,16450000.0,2280.0,7994.0,5618.0,4261.0,8412.0,5832.0
1646,16460000.0,2295.0,8189.0,5543.0,4238.0,8582.0,5796.0
1647,16470000.0,2191.0,7661.0,5485.0,4022.0,8034.0,5656.0
1648,16480000.0,2251.0,8111.0,5637.0,3996.0,8503.0,5891.0


In [18]:
df.groupby(['interval']).agg({
    'ro_shared_cacheline_count': 'sum',
    'rw_shared_cacheline_count': 'sum',
    'private_cacheline_count': 'sum',
    'ro_shared_access_count': 'sum',
    'rw_shared_access_count': 'sum',
    'private_access_count': 'sum',
}).reset_index().copy()['interval']

0              0.0
1          10000.0
2          20000.0
3          30000.0
4          40000.0
           ...    
1645    16450000.0
1646    16460000.0
1647    16470000.0
1648    16480000.0
1649    16490000.0
Name: interval, Length: 1650, dtype: float64